# Evaluar un Data Agent de Fabric con tu propia verdad de referencia

Notebook de apoyo a la charla *Diseño y operacionalización de Data Agents en Microsoft Fabric*.

Un Data Agent responde preguntas de negocio en lenguaje natural. Antes de fiarte de él, **ponlo a prueba**: hazle preguntas cuya respuesta correcta ya conoces y comprueba qué contesta. Este notebook lo hace en cuatro pasos:

1. **Calcular la verdad de referencia** con Spark SQL sobre las tablas del Lakehouse. Las respuestas esperadas salen de los propios datos, así que la prueba sigue siendo válida cuando los datos cambian.
2. **Montar** una tabla con `question` + `expected_answer`.
3. **Ejecutar la evaluación** con el SDK de Data Agent de Fabric, usando un *critic prompt* personalizado para que la comprobación sea sobre la corrección y no sobre el formato.
4. **Leer los resultados**: un resumen y el detalle paso a paso de cada pregunta.

### Requisitos

- Un workspace de Fabric con un **Data Agent publicado** conectado a las tablas de Olist (el nombre del agente se indica en el paso 3).
- El [dataset público de comercio electrónico brasileño de Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) cargado como tablas Delta en un Lakehouse: `customers`, `orders`, `order_items`, `order_payments`, `order_reviews`, `products`, `product_category_name_translation`, `sellers`. Consulta la página del dataset para ver su licencia. Los datos no se incluyen en este repositorio.
- Este notebook abierto en el mismo workspace, con ese Lakehouse como **Lakehouse predeterminado**.
- Documentación: [Evaluar tu data agent](https://learn.microsoft.com/es-es/fabric/data-science/evaluate-data-agent).

> Las salidas se han eliminado a propósito, para que el notebook no lleve nombres de workspace, identificadores ni historial de ejecuciones. Ejecútalo tú para ver los resultados.

In [ ]:
# Instala el SDK de Data Agent de Fabric (incluye las funciones de evaluación).
# Ejecútalo una vez por sesión y reinicia el kernel si el notebook lo pide.
%pip install -U fabric-data-agent-sdk

## Paso 1. Calcular la verdad de referencia a partir de los datos

Cada consulta responde una pregunta de negocio y guarda un único valor (o fila) en `results`. Mantener la lógica aquí, en lugar de escribir las respuestas esperadas a mano, hace que una respuesta esperada incorrecta sea un error que se puede leer y corregir.

In [ ]:
import pandas as pd

# Claves de las ocho comprobaciones siguientes, en el orden en que las preguntaremos.
QUESTIONS = [
    "avg_review_score",
    "best_category_by_review",
    "late_delivery_count",
    "revenue_2018",
    "state_with_most_sellers",
    "most_common_payment_type",
    "avg_installments_credit_card",
    "unique_customers",
]

results = {}

# 1. Puntuación media de las reseñas de todos los pedidos (una agregación simple).
results["avg_review_score"] = spark.sql("""
    SELECT ROUND(AVG(review_score), 2) AS v FROM order_reviews
""").first()["v"]

In [ ]:
# 2. Categoría de producto con la puntuación media de reseñas más alta.
#    Exigimos al menos 30 reseñas para que una categoría con una sola reseña
#    de 5 estrellas no gane. Esta pregunta necesita un join de varias tablas,
#    una buena prueba de si el agente entiende las relaciones.
results["best_category_by_review"] = spark.sql("""
    SELECT t.product_category_name_english AS category,
           ROUND(AVG(r.review_score), 2) AS avg_score,
           COUNT(*) AS n
    FROM order_reviews r
    JOIN order_items oi ON oi.order_id = r.order_id
    JOIN products p ON p.product_id = oi.product_id
    JOIN product_category_name_translation t
         ON t.product_category_name = p.product_category_name
    GROUP BY t.product_category_name_english
    HAVING COUNT(*) >= 30
    ORDER BY avg_score DESC
    LIMIT 1
""").first()

In [ ]:
# 3. Pedidos entregados después de la fecha de entrega estimada.
#    Se excluyen los pedidos sin fecha de entrega (no se entregaron).
results["late_delivery_count"] = spark.sql("""
    SELECT COUNT(*) AS v
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
      AND order_delivered_customer_date > order_estimated_delivery_date
""").first()["v"]

In [ ]:
# 4. Ingresos totales (precio del artículo + transporte) de los pedidos de 2018.
#    Un filtro de fecha más una suma: comprueba que el agente aplica el año
#    correcto e incluye el transporte, como dice la pregunta.
results["revenue_2018"] = spark.sql("""
    SELECT ROUND(SUM(oi.price + oi.freight_value), 2) AS v
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE YEAR(o.order_purchase_timestamp) = 2018
""").first()["v"]

In [ ]:
# 5. Estado con más vendedores distintos.
#    COUNT(DISTINCT ...) es importante: un vendedor aparece en muchas filas.
results["state_with_most_sellers"] = spark.sql("""
    SELECT seller_state AS state, COUNT(DISTINCT seller_id) AS n
    FROM sellers
    GROUP BY seller_state
    ORDER BY n DESC
    LIMIT 1
""").first()

In [ ]:
# 6. Tipo de pago más habitual.
results["most_common_payment_type"] = spark.sql("""
    SELECT payment_type, COUNT(*) AS n
    FROM order_payments
    GROUP BY payment_type
    ORDER BY n DESC
    LIMIT 1
""").first()

In [ ]:
# 7. Media de cuotas, solo pagos con tarjeta de crédito (agregación + filtro).
results["avg_installments_credit_card"] = spark.sql("""
    SELECT ROUND(AVG(payment_installments), 2) AS v
    FROM order_payments
    WHERE payment_type = 'credit_card'
""").first()["v"]

In [ ]:
# 8. Clientes únicos.
#    El dataset tiene un customer_id por pedido y un customer_unique_id por
#    persona. La pregunta es sobre personas, así que contamos el segundo.
#    Es un lugar clásico donde un agente puede elegir la columna equivocada.
results["unique_customers"] = spark.sql("""
    SELECT COUNT(DISTINCT customer_unique_id) AS v FROM customers
""").first()["v"]

## Paso 2. Montar el conjunto de preguntas y respuestas esperadas

El SDK espera un DataFrame con dos columnas: `question` (lo que preguntaría un usuario) y `expected_answer` (lo que debe contener una respuesta correcta). Las respuestas esperadas incluyen el número *y* la entidad, así que también se detecta una categoría o un estado equivocado.

Las preguntas están en español. Si tu agente está configurado en inglés, puedes usar las preguntas del notebook en inglés (`data_agent_evaluation_en.ipynb`).

In [ ]:
# --- Montar el conjunto pregunta / respuesta esperada ----------------------
eval_rows = [
    ("¿Cuál es la puntuación media de las reseñas de todos los pedidos?",
     f"{results['avg_review_score']}"),

    ("¿Qué categoría de producto tiene la puntuación media de reseñas más alta?",
     f"{results['best_category_by_review']['category']} "
     f"(media {results['best_category_by_review']['avg_score']}, "
     f"{results['best_category_by_review']['n']} reseñas)"),

    ("¿Cuántos pedidos se entregaron después de la fecha de entrega estimada?",
     f"{results['late_delivery_count']}"),

    ("¿Cuáles fueron los ingresos totales, incluido el transporte, de los pedidos realizados en 2018?",
     f"{results['revenue_2018']}"),

    ("¿Qué estado tiene más vendedores?",
     f"{results['state_with_most_sellers']['state']} "
     f"({results['state_with_most_sellers']['n']} vendedores)"),

    ("¿Cuál es el tipo de pago más habitual?",
     f"{results['most_common_payment_type']['payment_type']} "
     f"({results['most_common_payment_type']['n']} pagos)"),

    ("¿Cuál es el número medio de cuotas en los pagos con tarjeta de crédito?",
     f"{results['avg_installments_credit_card']}"),

    ("¿Cuántos clientes únicos hay?",
     f"{results['unique_customers']}"),
]

## Paso 3. Ejecutar la evaluación

Configura los parámetros siguientes y ejecuta. El SDK envía cada pregunta a tu agente, registra la respuesta y los pasos de razonamiento del agente, y pide a un modelo juez (el *critic*) que decida si la respuesta es correcta.

In [ ]:
df = pd.DataFrame(eval_rows, columns=["question", "expected_answer"])
display(df)

# --- ejecutar la evaluación ------------------------------------------
from fabric.dataagent.evaluation import evaluate_data_agent

# Nombre de tu Data Agent publicado (exactamente como aparece en el workspace).
data_agent_name = "<your-data-agent-name>"

# (Opcional) indícalo si el Data Agent está en otro workspace.
workspace_name = None

# Se crean dos tablas: "<table_name>" (resumen) y "<table_name>_steps"
# (razonamiento paso a paso y traza).
table_name = "olist_evaluation_output"

# "production" (por defecto, la versión publicada) o "sandbox" (sin publicar,
# para probar cambios antes de publicarlos).
data_agent_stage = "production"

### Un critic prompt personalizado

Las respuestas numéricas y de negocio rara vez coinciden carácter a carácter: el agente puede decir "R$ 1.234.567,89 de ingresos totales" mientras que nuestra respuesta esperada es `1234567.89`. Un critic prompt personalizado hace que la comprobación sea sobre la **corrección, no sobre el formato**. Ajusta las reglas de tolerancia a tu propio dominio.

El prompt se deja en inglés a propósito (así funciona igual que en la versión en inglés). Puedes traducirlo, pero vuelve a probar el resultado.

In [ ]:
# Las respuestas numéricas y de negocio rara vez coinciden carácter a carácter
# (el agente dirá "R$ 1.234.567,89 de ingresos totales" frente a nuestro
# "1234567.89"), así que un critic_prompt personalizado mantiene la
# comprobación centrada en la corrección y no en el formato.

critic_prompt = """
    Given the following query, expected answer, and actual answer, decide if
    the actual answer is CORRECT relative to the expected answer.

    Treat the answer as correct if:
    - Numbers match within reasonable rounding (e.g. currency formatting,
      1-2 decimal places, thousands separators).
    - The same entity/category/state is identified, even if phrased differently.

    Query: {query}

    Expected Answer:
    {expected_answer}

    Is the actual answer correct? Respond with 'yes' or 'no' and one short
    sentence explaining why.
"""

evaluation_id = evaluate_data_agent(
    df,
    data_agent_name,
    workspace_name=workspace_name,
    table_name=table_name,
    data_agent_stage=data_agent_stage,
    critic_prompt=critic_prompt,
)

In [ ]:
# Guarda este id: identifica esta ejecución en las tablas de salida.
print(f"Evaluation run ID: {evaluation_id}")

## Paso 4. Leer los resultados

Empieza por el resumen (cuántas respuestas se consideraron correctas) y abre después el detalle de las preguntas que fallaron o quedaron poco claras. La tabla de detalle incluye los pasos de razonamiento del agente, que suelen mostrar *por qué* se equivocó (tabla equivocada, columna equivocada, una instrucción que falta).

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_summary

# Una fila por ejecución: cuántas respuestas fueron true, false o unclear.
summary_df = get_evaluation_summary(table_name, verbose=True)
display(summary_df)

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_details

details_df = get_evaluation_details(
    evaluation_id,
    table_name,
    get_all_rows=True,  # get_all_rows=False -> solo las filas false/unclear, las que
                        # conviene leer primero cuando depuras el agente
    verbose=True,
)
display(details_df)

## Qué hacer ante un fallo

- **Columna o tabla equivocada:** añade una descripción más clara de la tabla o columna, o una instrucción al agente ("clientes significa `customer_unique_id`, no `customer_id`").
- **Lógica equivocada:** añade una consulta de ejemplo para ese tipo de pregunta a las consultas de ejemplo del agente.
- **Respuesta correcta con otro formato:** relaja el critic prompt, no el agente.
- Vuelve a ejecutar con `data_agent_stage = "sandbox"` tras cada cambio y publica solo cuando la ejecución salga limpia.